In [9]:
import os
import pandas as pd
import numpy as np
import json
import re

from sklearn.metrics import f1_score, accuracy_score

from sentence_transformers import SentenceTransformer
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from scipy.spatial.distance import cosine
from typing import Dict, List 

In [10]:
ground_truth=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/BIDS_data/anotated_processed.csv")
results_1_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_1/ovis2/20260113_1449/predictions.csv")
results_1_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_1/qwen25/20260113_1943/predictions.csv")
results_2_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_2/ovis2/20260113_1449/predictions.csv")
results_2_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_2/qwen25/20260113_1527/predictions.csv")
results_3_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_3/ovis2/20260113_1450/predictions.csv")
results_3_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_3/qwen25/20260113_1812/predictions.csv")

In [11]:
# -----------------------
# 1) Normalization (keeps "na"/"n_a"/"n/a" as a real label -> "na")
# -----------------------
MISSING_STRINGS = {"", "none", "null"}
NA_LABEL_STRINGS = {"na", "n/a", "n_a"}

def norm_text(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()

    # treat n/a variants as a meaningful label (not missing)
    if s in NA_LABEL_STRINGS:
        return "na"

    if s in MISSING_STRINGS:
        return pd.NA

    # unify underscores/spaces like before
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -----------------------
# 2) Parse JSON that may be wrapped in ```json ... ```
# -----------------------
CODEBLOCK_RE = re.compile(r"^```(?:json)?\s*(.*?)\s*```$", re.DOTALL | re.IGNORECASE)


def safe_load(s):
    # Always return a dict
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return {}

    # if it's already parsed
    if isinstance(s, dict):
        return s
    if isinstance(s, list):
        # if it's a list of dicts, take first; otherwise treat as empty
        return s[0] if (len(s) > 0 and isinstance(s[0], dict)) else {}

    txt = str(s).strip()

    # remove ```json ... ``` fences
    m = CODEBLOCK_RE.match(txt)
    if m:
        txt = m.group(1).strip()

    # remove stray "json" prefix if present
    txt = re.sub(r"^\s*json\s*", "", txt, flags=re.IGNORECASE).strip()

    try:
        obj = json.loads(txt)
    except Exception:
        return {}

    # json.loads might return list; normalize to dict
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        return obj[0] if (len(obj) > 0 and isinstance(obj[0], dict)) else {}

    return {}

# -----------------------
# 3) Build prediction table from results
# -----------------------
def analyse_df(results_df,GT_COLS,PRED_TO_GT={},TEXT_COLS={},NUM_COLS={}):
    pred = pd.json_normalize(results_df["raw_prediction"].map(safe_load))
    pred["video_path"] = results_df["video_path"].values

    print(PRED_TO_GT)
    pred = pred.rename(columns=PRED_TO_GT)
    pred = pred[["video_path"] + GT_COLS].copy()
    # ensure all expected cols exist
    for c in GT_COLS:
        if c not in pred.columns:
            pred[c] = pd.NA
    # normalize predictions
    for c in GT_COLS:
        pred[c] = pred[c].map(norm_text)

    for c in TEXT_COLS:
        pred[c] = pred[c].map(norm_text)
    for c in NUM_COLS:
        pred[c] = pred[c].map(norm_num).astype("Int64")


    locomotion_map = {
        "crawling": "crawl",
        "cruising": "cruise",
        "walking": "walk",
        "running": "run",
        "multiple": "multiple",
        "vehicle": "vehicle",
    }
    if "Locomotion_type" in pred.columns:
        pred["Locomotion_type"] = (
            pred["Locomotion_type"]
              .astype("string")
              .str.strip()
              .str.lower()
              .str.replace("_", " ", regex=False)   # so n_a -> n a
              .replace(locomotion_map)
              .replace({"n a": pd.NA, "na": pd.NA, "n/a": pd.NA})  # make n_a become missing for this field
        )
    if "Support_type" in pred.columns:
        # fix the weird value in predictions
        pred["Support_type"] = (
            pred["Support_type"]
              .astype("string")
              .str.strip()
              .replace({"verbal and physical": "both"})
        )
    # -----------------------
    # 4) Prepare ground truth (assumes column names already match)

    # -----------------------
    GT_PATH_COL = "BidsProcessed"   # change if ground truth path column is different for this task
    
    gt = ground_truth.copy()
    for c in GT_COLS:
        if c not in gt.columns:
            raise KeyError(f"ground_truth is missing column: {c}")
        gt[c] = gt[c].map(norm_text)
    
    for c in TEXT_COLS:
        gt[c] = gt[c].map(norm_text)
    for c in NUM_COLS:
        gt[c] = gt[c].map(norm_num).astype("Int64")

    # -----------------------
    # 5) Merge
    # -----------------------

    merged = pred.merge(
        gt[[GT_PATH_COL] + GT_COLS],
        left_on="video_path",
        right_on=GT_PATH_COL,
        how="left",
        suffixes=("_pred", "_gt"),
        indicator=True
    )
    
    # -----------------------
    # 6) Match columns + Accuracy (only where GT exists)
    # -----------------------
    for c in GT_COLS:
        both_na = merged[f"{c}_pred"].isna() & merged[f"{c}_gt"].isna()
        merged[f"{c}_match"] = (merged[f"{c}_pred"] == merged[f"{c}_gt"]) | both_na
    
    acc = {}
    n_eval = {}
    for c in GT_COLS:
        mask = (merged["_merge"] == "both") & merged[f"{c}_gt"].notna()
        n = int(mask.sum())
        n_eval[c] = n
        acc[c] = float(merged.loc[mask, f"{c}_match"].sum() / n) if n > 0 else np.nan
    
    acc = pd.Series(acc).sort_values(ascending=False)
    n_eval = pd.Series(n_eval)
    
    print("Rows with no matching GT (by merge key):")
    print(merged.loc[merged["_merge"] != "both", "video_path"].head(20))
    
    print("\nDenominators (n evaluated) per field:")
    print(n_eval)
    
    print("\nAccuracy per field:")
    print(acc)
    return pred, gt, merged
    
    # -----------------------
    # 7) Macro/Weighted F1 for categorical fields
    # -----------------------
def f1_metrics(df, col):
    mask = (df["_merge"] == "both") & df[f"{col}_gt"].notna()
    if mask.sum() == 0:
        return {"n": 0, "accuracy": np.nan, "macro_f1": np.nan, "weighted_f1": np.nan}

    y_true = df.loc[mask, f"{col}_gt"].astype("string")
    y_pred = df.loc[mask, f"{col}_pred"].astype("string").fillna("__MISSING__")

    return {
        "n": int(mask.sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }




In [12]:
def norm_text(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()
    if s.lower() in NA_STRINGS:
        return pd.NA

    s = s.lower()

    # Make underscores and whitespace equivalent:
    # 1) underscores -> spaces
    s = s.replace("_", " ")

    # 2) collapse multiple spaces/tabs/newlines
    s = re.sub(r"\s+", " ", s).strip()

    return s

def norm_num(x):
    x = norm_text(x)
    if x is pd.NA:
        return pd.NA
    try:
        return int(float(x))
    except Exception:
        return pd.NA
def safe_load(s):
    # Always return a dict
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return {}

    # if it's already parsed
    if isinstance(s, dict):
        return s
    if isinstance(s, list):
        # if it's a list of dicts, take first; otherwise treat as empty
        return s[0] if (len(s) > 0 and isinstance(s[0], dict)) else {}

    txt = str(s).strip()

    # remove ```json ... ``` fences
    m = CODEBLOCK_RE.match(txt)
    if m:
        txt = m.group(1).strip()

    # remove stray "json" prefix if present
    txt = re.sub(r"^\s*json\s*", "", txt, flags=re.IGNORECASE).strip()

    try:
        obj = json.loads(txt)
    except Exception:
        return {}

    # json.loads might return list; normalize to dict
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        return obj[0] if (len(obj) > 0 and isinstance(obj[0], dict)) else {}

    return {}

# --- 1) Build prediction annotations table ---
def accuracy_annotations_1(results_1):
    pred_anno = pd.json_normalize(results_1["prediction"].map(safe_load))
    pred_anno["video_path"] = results_1["video_path"].values
    
    # rename differing columns + keep only what we need
    pred_anno = pred_anno.rename(columns=PRED_TO_GT)
    
    # Ensure all GT_COLS exist in pred_anno (in case some keys missing)
    for c in GT_COLS:
        if c not in pred_anno.columns:
            pred_anno[c] = pd.NA
    
    pred_anno = pred_anno[["video_path"] + GT_COLS].copy()
    
    # --- 2) Normalize types/strings for fair comparison ---
    for c in TEXT_COLS:
        pred_anno[c] = pred_anno[c].map(norm_text)
    for c in NUM_COLS:
        pred_anno[c] = pred_anno[c].map(norm_num).astype("Int64")
    
    gt = ground_truth.copy()
    
    for c in GT_COLS:
        if c not in gt.columns:
            raise KeyError(f"ground_truth is missing column: {c}")
    
    for c in TEXT_COLS:
        gt[c] = gt[c].map(norm_text)
    for c in NUM_COLS:
        gt[c] = gt[c].map(norm_num).astype("Int64")

    
    # --- 3) Merge (exact path match) ---
    merged = pred_anno.merge(
        gt[["BidsProcessed"] + GT_COLS],
        left_on="video_path",
        right_on="BidsProcessed",
        how="left",
        suffixes=("_pred", "_gt"),
        indicator=True
    )
        
    # --- 4) Compare every field ---
    for c in GT_COLS:
        both_na = merged[f"{c}_pred"].isna() & merged[f"{c}_gt"].isna()
        merged[f"{c}_match"] = (merged[f"{c}_pred"] == merged[f"{c}_gt"]) | both_na
    
    acc = {}
    n_eval = {}
    for c in GT_COLS:
        mask = (merged["_merge"] == "both") & merged[f"{c}_gt"].notna()
        n = int(mask.sum())
        n_eval[c] = n
        acc[c] = float(merged.loc[mask, f"{c}_match"].sum() / n) if n > 0 else np.nan
        
    print("\nDenominators (n evaluated) per field:")
    print(n_eval)
    
    print("Rows with no matching GT (by merge key):")
    print(merged.loc[merged["_merge"] != "both", "video_path"].head(20))
    
    print("\nPer-field match rate (excluding rows where GT is missing for that field):")
    print(acc)
    
    # Optional: get a dataframe of mismatches (only where a GT row was found)
    match_cols = [f"{c}_match" for c in GT_COLS]
    mismatches = merged.loc[
        (merged["_merge"] == "both") & (~merged[match_cols].all(axis=1)),
        ["video_path"] + sum([[f"{c}_pred", f"{c}_gt", f"{c}_match"] for c in GT_COLS], [])
    ]
    return pred_anno,gt,merged


In [13]:

def compute_f1_metrics(merged: pd.DataFrame, col: str) -> dict:
    y_true = merged[f"{col}_gt"]
    y_pred = merged[f"{col}_pred"]

    # Evaluate only where GT exists for that field
    mask = y_true.notna()

    if mask.sum() == 0:
        return {"n": 0, "accuracy": np.nan, "macro_f1": np.nan, "weighted_f1": np.nan}

    yt = y_true[mask].astype("string")
    yp = y_pred[mask].astype("string")

    return {
        "n": int(mask.sum()),
        "accuracy": float(accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, average="macro")),
        "weighted_f1": float(f1_score(yt, yp, average="weighted")),
    }



In [21]:
class Evaluator:
    """
    Computes lexical metrics and semantic similarity metrics 
    Attributes:
        rouge: ROUGE metric calculator.
        embedding_model: SentenceTransformer for semantic similarity.
        embedding_model_name: Name of the embedding model.
    """
    
    def __init__(self, embedding_model_name: str = 'all-MiniLM-L6-v2'):
        """Initializes evaluator with embedding model.
        
        Args:
            embedding_model_name: Name of sentence-transformers model.
                Options:
                - 'all-MiniLM-L6-v2': Fast, 384-dim (default)
                - 'all-mpnet-base-v2': Higher quality, 768-dim
                - 'paraphrase-MiniLM-L6-v2': Paraphrase detection
        """
        self.rouge = Rouge()
        print(f"Loading embedding model: {embedding_model_name}")
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.embedding_model_name = embedding_model_name
    
    def _compute_word_overlap(self, pred: str, truth: str) -> float:
        """Computes word overlap ratio between two texts.
        
        Args:
            pred: Predicted text.
            truth: Ground truth text.
            
        Returns:
            Overlap ratio (0.0 to 1.0).
        """
        pred_words = set(pred.lower().split())
        truth_words = set(truth.lower().split())
        
        if not truth_words:
            return 0.0
        
        overlap = len(pred_words & truth_words)
        return overlap / len(truth_words)
    
    def _compute_bleu(self, pred: str, truth: str) -> float:
        """Computes BLEU score with smoothing.
        
        Args:
            pred: Predicted text.
            truth: Ground truth text.
            
        Returns:
            BLEU score (0.0 to 1.0).
        """
        reference = [truth.split()]
        candidate = pred.split()
        smoothing = SmoothingFunction().method1
        
        try:
            return sentence_bleu(
                reference, 
                candidate, 
                smoothing_function=smoothing
            )
        except:
            return 0.0
    
    def _compute_rouge(self, pred: str, truth: str) -> Dict[str, float]:
        """Computes ROUGE scores.
        
        Args:
            pred: Predicted text.
            truth: Ground truth text.
            
        Returns:
            Dictionary with rouge-1, rouge-2, rouge-l F1 scores.
        """
        try:
            scores = self.rouge.get_scores(pred, truth)[0]
            return {
                'rouge1': scores['rouge-1']['f'],
                'rouge2': scores['rouge-2']['f'],
                'rougeL': scores['rouge-l']['f']
            }
        except:
            return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    
    def _compute_semantic_similarity(
        self, 
        predictions: List[str], 
        ground_truths: List[str]
    ) -> Dict[str, float]:
        """Computes semantic similarity metrics using embeddings.
        
        Args:
            predictions: List of predicted texts.
            ground_truths: List of ground truth texts.
            
        Returns:
            Dictionary with cosine similarity, euclidean distance, and dot product.
        """
        # Filter valid pairs
        valid_pairs = [
            (str(pred), str(truth))
            for pred, truth in zip(predictions, ground_truths)
            if pred and truth and not pd.isna(pred) and not pd.isna(truth)
        ]
        
        if not valid_pairs:
            return {
                'cosine_similarity': 0.0,
                'euclidean_distance': 0.0,
                'dot_product': 0.0,
                'count': 0
            }
        
        pred_texts, truth_texts = zip(*valid_pairs)
        
        # Generate embeddings
        pred_embeddings = self.embedding_model.encode(
            list(pred_texts), 
            show_progress_bar=False,
            convert_to_numpy=True
        )
        truth_embeddings = self.embedding_model.encode(
            list(truth_texts),
            show_progress_bar=False,
            convert_to_numpy=True
        )
        
        # Compute metrics
        cosine_sims = [
            1 - cosine(pred_emb, truth_emb)
            for pred_emb, truth_emb in zip(pred_embeddings, truth_embeddings)
        ]
        
        euclidean_dists = [
            np.linalg.norm(pred_emb - truth_emb)
            for pred_emb, truth_emb in zip(pred_embeddings, truth_embeddings)
        ]
        
        dot_products = [
            np.dot(pred_emb, truth_emb)
            for pred_emb, truth_emb in zip(pred_embeddings, truth_embeddings)
        ]
        
        return {
            'cosine_similarity': float(np.mean(cosine_sims)),
            'euclidean_distance': float(np.mean(euclidean_dists)),
            'dot_product': float(np.mean(dot_products)),
            'count': len(valid_pairs)
        }
    def evaluate_activities(
        self, 
        predictions: List[str], 
        ground_truths: List[str]
    ) -> Dict:
        """Evaluates activity predictions with lexical and semantic metrics.
        
        Args:
            predictions: List of predicted activity descriptions.
            ground_truths: List of ground truth descriptions.
            
        Returns:
            Dictionary with all activity metrics.
        """
        # Filter valid pairs - handle pandas NA properly
        valid_pairs = []
        for pred, truth in zip(predictions, ground_truths):
            # Skip if either is actually NA/NaN
            if pd.isna(pred) or pd.isna(truth):
                continue
                
            # Convert to string
            pred_str = str(pred).strip()
            truth_str = str(truth).strip()
            
            # Skip only if empty string or the string literal "nan"
            # Keep "none" as it's a valid category value for Constraint_type
            if (pred_str == '' or truth_str == '' or 
                pred_str.lower() == 'nan' or truth_str.lower() == 'nan'):
                continue
                
            valid_pairs.append((pred_str, truth_str))
        
        if not valid_pairs:
            return {
                'bleu': 0.0,
                'rouge1': 0.0,
                'rouge2': 0.0,
                'rougeL': 0.0,
                'word_overlap': 0.0,
                'cosine_similarity': 0.0,
                'euclidean_distance': 0.0,
                'dot_product': 0.0,
                'count': 0
            }
        
        pred_texts, truth_texts = zip(*valid_pairs)
        
        # Lexical metrics
        bleu_scores = []
        rouge_scores = []
        word_overlaps = []
        
        for pred, truth in valid_pairs:
            bleu_scores.append(self._compute_bleu(pred, truth))
            rouge_scores.append(self._compute_rouge(pred, truth))
            word_overlaps.append(self._compute_word_overlap(pred, truth))
        
        # Average ROUGE scores
        avg_rouge = {
            'rouge1': np.mean([s['rouge1'] for s in rouge_scores]),
            'rouge2': np.mean([s['rouge2'] for s in rouge_scores]),
            'rougeL': np.mean([s['rougeL'] for s in rouge_scores])
        }
        
        # Semantic similarity metrics
        semantic_metrics = self._compute_semantic_similarity(
            list(pred_texts), 
            list(truth_texts)
        )
        
        return {
            # Lexical metrics
            'bleu': float(np.mean(bleu_scores)),
            'rouge1': avg_rouge['rouge1'],
            'rouge2': avg_rouge['rouge2'],
            'rougeL': avg_rouge['rougeL'],
            'word_overlap': float(np.mean(word_overlaps)),
            # Semantic metrics
            'cosine_similarity': semantic_metrics['cosine_similarity'],
            'euclidean_distance': semantic_metrics['euclidean_distance'],
            'dot_product': semantic_metrics['dot_product'],
            'count': len(valid_pairs)
        }

def evaluate_free_text_field(merged_df, field_name, evaluator):
    """Evaluate a free-text field using lexical and semantic metrics."""
    mask = (merged_df["_merge"] == "both") & merged_df[f"{field_name}_gt"].notna()
    
    predictions = merged_df.loc[mask, f"{field_name}_pred"].tolist()
    ground_truths = merged_df.loc[mask, f"{field_name}_gt"].tolist()
    
    metrics = evaluator.evaluate_activities(predictions, ground_truths)
    
    return pd.Series(metrics, name=field_name)
    

## Annotations part 1

In [22]:
GT_COLS = [
    "Context", "Location", "Activity", "Child_of_interest_clear",
    "#_adults", "#_children", "#_people_background",
    "Interaction_with_child", "#_people_interacting",
    "Child_constrained", "Constraint_type",
    "Supports", "Support_type", "Example_support_type"
]

# Map prediction keys -> ground_truth keys where they differ
PRED_TO_GT = {
    "Num_adults_foreground": "#_adults",
    "Num_children_foreground": "#_children",
    "Num_people_background": "#_people_background",
    "Num_people_interacting_with_child": "#_people_interacting",
}

NUM_COLS = ["#_adults", "#_children", "#_people_background", "#_people_interacting"]
TEXT_COLS = [c for c in GT_COLS if c not in NUM_COLS]

NA_STRINGS = {"n/a", "na", "none", "null", ""}
F1_COLS = [
    "Interaction_with_child",
    "Child_constrained",
    "Supports","Child_of_interest_clear"

]


In [23]:
print("Prediction Evaluation for the first split of annotations with Ovis2\n")
pred_1_ovis,gt_1_ovis,merged_1_ovis=analyse_df(results_1_ovis,GT_COLS,PRED_TO_GT,TEXT_COLS,NUM_COLS)
metrics = []
for c in F1_COLS:
    m = compute_f1_metrics(merged_1_ovis, c)
    metrics.append({"field": c, **m})

metrics_df_1_ovis = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
print(metrics_df_1_ovis)

# Evaluate Activity and Constraint_type
evaluator = Evaluator(embedding_model_name='all-MiniLM-L6-v2')
activity_metrics_ovis = evaluate_free_text_field(merged_1_ovis, "Activity", evaluator)
Constraint_type_metrics_ovis = evaluate_free_text_field(merged_1_ovis, "Constraint_type", evaluator)

print(activity_metrics_ovis)
print(Constraint_type_metrics_ovis)


Prediction Evaluation for the first split of annotations with Ovis2

{'Num_adults_foreground': '#_adults', 'Num_children_foreground': '#_children', 'Num_people_background': '#_people_background', 'Num_people_interacting_with_child': '#_people_interacting'}
Rows with no matching GT (by merge key):
Series([], Name: video_path, dtype: object)

Denominators (n evaluated) per field:
Context                    3315
Location                   3315
Activity                   3314
Child_of_interest_clear    3313
#_adults                   3312
#_children                 3309
#_people_background        3244
Interaction_with_child     3315
#_people_interacting       3275
Child_constrained          3314
Constraint_type             731
Supports                   3315
Support_type               2295
Example_support_type       2296
dtype: int64

Accuracy per field:
Location                   0.884766
#_people_background        0.866831
Child_of_interest_clear    0.858135
#_children                 0.

In [25]:
print("Prediction Evaluation for the first split of annotations with Qwen2.5\n")
pred_1_qwen,gt_1_qwen,merged_1_qwen=analyse_df(results_1_qwen,GT_COLS,PRED_TO_GT,TEXT_COLS,NUM_COLS)
metrics = []
for c in F1_COLS:
    m = compute_f1_metrics(merged_1_qwen, c)
    metrics.append({"field": c, **m})

metrics_df_1_qwen = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
activity_metrics_qwen = evaluate_free_text_field(merged_1_qwen, "Activity", evaluator)
Constraint_type_metrics_qwen = evaluate_free_text_field(merged_1_qwen, "Constraint_type", evaluator)
print(activity_metrics_qwen)
print(Constraint_type_metrics_qwen)
print(metrics_df_1_qwen)

Prediction Evaluation for the first split of annotations with Qwen2.5

{'Num_adults_foreground': '#_adults', 'Num_children_foreground': '#_children', 'Num_people_background': '#_people_background', 'Num_people_interacting_with_child': '#_people_interacting'}
Rows with no matching GT (by merge key):
Series([], Name: video_path, dtype: object)

Denominators (n evaluated) per field:
Context                    3315
Location                   3315
Activity                   3314
Child_of_interest_clear    3313
#_adults                   3312
#_children                 3309
#_people_background        3244
Interaction_with_child     3315
#_people_interacting       3275
Child_constrained          3314
Constraint_type             731
Supports                   3315
Support_type               2295
Example_support_type       2296
dtype: int64

Accuracy per field:
Child_of_interest_clear    0.891639
#_people_background        0.867139
#_children                 0.854337
Location                   

## Annotations part 2

In [30]:
# -----------------------
GT2_COLS = [
    "Gestures", "Gesture_type", "Vocalizations",
    "RMM", "RMM_type", "Response_to_name",
    "Locomotion", "Locomotion_type",
    "Grasping", "Grasp_type"
]

F1_2_COLS = [
    "Gestures", "Vocalizations",
    "RMM",  "Response_to_name",
    "Locomotion", 
    "Grasping", 
]
print("Prediction Evaluation for the second split of annotations with Ovis2\n")
pred_2_ovis,gt_2_ovis,merged_2_ovis=analyse_df(results_2_ovis,GT_COLS=GT2_COLS)
metrics = []
for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_ovis, c)
    metrics.append({"field": c, **m})

metrics_df_2_ovis= pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
metrics_df_2_ovis

Prediction Evaluation for the second split of annotations with Ovis2

{}
Rows with no matching GT (by merge key):
Series([], Name: video_path, dtype: object)

Denominators (n evaluated) per field:
Gestures            3313
Gesture_type        1607
Vocalizations       3314
RMM                 3314
RMM_type             372
Response_to_name     227
Locomotion          3311
Locomotion_type     1420
Grasping            3314
Grasp_type          1916
dtype: int64

Accuracy per field:
RMM                 0.842788
Grasping            0.668075
Locomotion          0.622471
Gestures            0.584365
Grasp_type          0.507829
Vocalizations       0.264937
Gesture_type        0.212197
Locomotion_type     0.093662
RMM_type            0.040323
Response_to_name    0.013216
dtype: float64


,n,accuracy,macro_f1,weighted_f1
field,,,,
RMM,3314,0.842788,0.551342,0.831768
Grasping,3314,0.668075,0.660902,0.668402
Gestures,3313,0.584365,0.584146,0.584477
Locomotion,3311,0.622471,0.405363,0.536408
Vocalizations,3314,0.264937,0.216749,0.122483
Response_to_name,227,0.013216,0.013636,0.025230


In [32]:
print("Prediction Evaluation for the second split of annotations with Qwen2.5\n")

pred_2_qwen,gt_2_qwen,merged_2_qwen=analyse_df(results_2_qwen,GT_COLS=GT2_COLS)
metrics = []

for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_qwen, c)
    metrics.append({"field": c, **m})

metrics_df_qwen= pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
metrics_df_qwen

Prediction Evaluation for the second split of annotations with Qwen2.5

{}
Rows with no matching GT (by merge key):
Series([], Name: video_path, dtype: object)

Denominators (n evaluated) per field:
Gestures            3313
Gesture_type        1607
Vocalizations       3314
RMM                 3314
RMM_type             372
Response_to_name     227
Locomotion          3311
Locomotion_type     1420
Grasping            3314
Grasp_type          1916
dtype: int64

Accuracy per field:
RMM                 0.884731
Locomotion          0.599215
Gestures            0.554482
Grasping            0.471032
Vocalizations       0.347013
Gesture_type        0.076540
Grasp_type          0.069415
Locomotion_type     0.048592
Response_to_name    0.017621
RMM_type            0.000000
dtype: float64


,n,accuracy,macro_f1,weighted_f1
field,,,,
RMM,3314,0.884731,0.489442,0.837828
Gestures,3313,0.554482,0.497574,0.503443
Locomotion,3311,0.599215,0.291537,0.479661
Grasping,3314,0.471032,0.390749,0.357115
Vocalizations,3314,0.347013,0.333492,0.287432
Response_to_name,227,0.017621,0.018393,0.033414


In [16]:
len(gt_2_qwen[gt_2_qwen["Vocalizations"]=="yes"])/len(gt_2_qwen[~gt_2_qwen["Vocalizations"].isna()])

0.7426071213035607

In [ ]:
set(pred_2_qwen["Locomotion_type"])

## Annotations part 3

In [25]:
GT3_TEXT_COLS = ["Body_Parts_Visible", "Angle_of_Body"]
GT3_NUM_COLS = [
    "Video_Quality_Child_Face_Visibility",
    "Video_Quality_Child_Body_Visibility",
    "Video_Quality_Child_Hand_Visibility",
    "Video_Quality_Lighting",
    "Video_Quality_Resolution",
    "Video_Quality_Motion",
]
GT3_COLS = GT3_TEXT_COLS + GT3_NUM_COLS


In [26]:
print("Prediction Evaluation for the third split of annotations with Ovis2\n")
pred_3_ovis,gt_3_ovis,merged_3_ovis=analyse_df(results_3_ovis,GT_COLS=GT3_COLS,TEXT_COLS=GT3_TEXT_COLS,NUM_COLS=GT3_NUM_COLS)

Prediction Evaluation for the third split of annotations with Ovis2

{}
Rows with no matching GT (by merge key):
Series([], Name: video_path, dtype: object)

Denominators (n evaluated) per field:
Body_Parts_Visible                     3311
Angle_of_Body                          3311
Video_Quality_Child_Face_Visibility    3314
Video_Quality_Child_Body_Visibility    3313
Video_Quality_Child_Hand_Visibility    3314
Video_Quality_Lighting                 3314
Video_Quality_Resolution               3314
Video_Quality_Motion                   3314
dtype: int64

Accuracy per field:
Body_Parts_Visible                     0.763214
Angle_of_Body                          0.405316
Video_Quality_Child_Body_Visibility    0.212798
Video_Quality_Child_Face_Visibility    0.149366
Video_Quality_Child_Hand_Visibility    0.148159
Video_Quality_Motion                   0.054315
Video_Quality_Lighting                 0.018105
Video_Quality_Resolution               0.002414
dtype: float64


In [27]:
print("Prediction Evaluation for the third split of annotations with Qwen2.5\n")
pred_3_qwen,gt_3_qwen,merged_3_qwen=analyse_df(results_3_qwen,GT_COLS=GT3_COLS,TEXT_COLS=GT3_TEXT_COLS,NUM_COLS=GT3_NUM_COLS)

Prediction Evaluation for the third split of annotations with Qwen2.5

{}
Rows with no matching GT (by merge key):
Series([], Name: video_path, dtype: object)

Denominators (n evaluated) per field:
Body_Parts_Visible                     3311
Angle_of_Body                          3311
Video_Quality_Child_Face_Visibility    3314
Video_Quality_Child_Body_Visibility    3313
Video_Quality_Child_Hand_Visibility    3314
Video_Quality_Lighting                 3314
Video_Quality_Resolution               3314
Video_Quality_Motion                   3314
dtype: int64

Accuracy per field:
Body_Parts_Visible                     0.623679
Angle_of_Body                          0.289943
Video_Quality_Child_Body_Visibility    0.233927
Video_Quality_Child_Face_Visibility    0.210320
Video_Quality_Child_Hand_Visibility    0.138202
Video_Quality_Resolution               0.090525
Video_Quality_Lighting                 0.089016
Video_Quality_Motion                   0.062764
dtype: float64
